#### 1. Librerías.

In [1]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [2]:
%run "./constantes/constantes.ipynb"

#### 3. Funciones.

In [3]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [4]:
#a. Conexión a la BBDD.
conn = sqlite3.connect(path_db)

In [5]:
#b. Leo las tablas de la BBDD como dataframes de pandas.
#i. Veo que tablas hay en la BBDD.
print("Tablas existentes:")
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tablas)
#ii. Leo las tablas.
print("\nLeyendo las tablas...")
try:
    df_libros = pd.read_sql_query("SELECT * FROM libros;", conn)
    df_lectores = pd.read_sql_query("SELECT * FROM lectores;", conn)
    df_interacciones = pd.read_sql_query("SELECT * FROM interacciones;", conn)
except Exception as e:
    print(f"Error al leer las tablas: {e}")
else:
    print("Tablas leídas correctamente.")
#iii. Cierro la conexión a la BBDD.
conn.close()

Tablas existentes:
            name
0         libros
1       lectores
2  interacciones

Leyendo las tablas...
Tablas leídas correctamente.


In [6]:
#c. Ejemplo a entregar, con los legajos a predecir su top 20 de libros a recomendar.
df_entregar = pd.read_csv("./inputs/ejemplo.csv")

#### 5. Calidad de datos.

In [7]:
#a. Cantidad de filas y columnas.
print("La tabla de libros, la cantidad de filas y columnas es: {}".format(df_libros.shape))
print("La tabla de lectores, la cantidad de filas y columnas es: {}".format(df_lectores.shape))
print("La tabla de interacciones, la cantidad de filas y columnas es: {}".format(df_interacciones.shape))

La tabla de libros, la cantidad de filas y columnas es: (128743, 9)
La tabla de lectores, la cantidad de filas y columnas es: (11285, 5)
La tabla de interacciones, la cantidad de filas y columnas es: (461408, 4)


In [8]:
#b. Me fijo si hay registros duplicados.
#i. Libros.
cant_libros_duplicados = df_libros["id_libro"].duplicated().sum()
print("Hay {} libros repetidos.".format(cant_libros_duplicados))
#ii. Usuarios.
cant_lectores_duplicados = df_lectores["id_lector"].duplicated().sum()
print("Hay {} usuarios repetidos.".format(cant_lectores_duplicados))
#iii. Combinaciones id_lector-id_libro en interacciones.
canti_interacciones_duplicados = df_interacciones[["id_libro","id_lector"]].duplicated().sum()
print("Hay {} lectores-libros repetidos.".format(canti_interacciones_duplicados))

Hay 0 libros repetidos.
Hay 0 usuarios repetidos.
Hay 0 lectores-libros repetidos.


In [9]:
#c. Me fijo la cantidad de usuarios y libros con interacciones.
#i. Libros.
cantidad_libros = len(df_libros["id_libro"].unique())
cantidad_libros_opiniones = len(df_interacciones["id_libro"].unique())
print("Hay {} libros totales, el {}% tienen opiniones.".format(cantidad_libros,round(cantidad_libros_opiniones/cantidad_libros*100,2)))
#ii. Lectores.
cantidad_lectores = len(df_lectores["id_lector"].unique())
cantidad_lectores_opiniones = len(df_interacciones["id_lector"].unique())
print("Hay {} lectores totales, el {}% otorgó opiniones.".format(cantidad_lectores,round(cantidad_lectores_opiniones/cantidad_lectores*100,2)))

Hay 128743 libros totales, el 37.39% tienen opiniones.
Hay 11285 lectores totales, el 94.58% otorgó opiniones.


In [11]:
#d. Cantidad y % de nulos por columna de cada tabla.
#i. Libros.
id_libros_unicos_con_opiniones = list(df_interacciones["id_libro"].unique())
libros_con_opiniones = df_libros[df_libros["id_libro"].isin(id_libros_unicos_con_opiniones)]
resumen_nulos_libros_con_interacciones = pd.DataFrame({
    "nulos": libros_con_opiniones.isnull().sum(),
    "pct_nulos": (libros_con_opiniones.isnull().sum() / len(libros_con_opiniones) * 100).round(2)
})

resumen_nulos_libros_con_interacciones

,nulos,pct_nulos
id_libro,0,0.00
titulo,0,0.00
autor,4,0.01
genero,0,0.00
editorial,8,0.02
anio_edicion,0,0.00
isbn,0,0.00
resumen,41,0.09
img_src,0,0.00


In [14]:
libros_con_opiniones[libros_con_opiniones["autor"].isna()]

,id_libro,titulo,autor,genero,editorial,anio_edicion,isbn,resumen,img_src
34988,el-principe-feliz,EL PRÍNCIPE FELIZ,None,Infantil y juvenil,EVEREST,2013,9788444149776.0,,https://quelibroleo.com/images/libros/84441497...
34998,simbad-el-marino-1,SIMBAD EL MARINO,None,Infantil y juvenil,EVEREST,2013,9788444149981.0,,https://quelibroleo.com/images/libros/84441499...
34999,el-traje-nuevo-del-emperador-2-1,EL TRAJE NUEVO DEL EMPERADOR,None,Infantil y juvenil,EVEREST,2013,9788444149813.0,,https://quelibroleo.com/images/libros/84441498...
42336,el-dhammapada-la-senda-de-la-perfeccion,EL DHAMMAPADA. La senda de la perfección,None,Clásicos de la literatura,PLATAFORMA,2014,9788415880998.0,El Dhammapada es una colección de 423 aforismo...,https://quelibroleo.com/images/libros/libro_14...


In [ ]:
df_libros[df_libros["id_libro"] == "el-juego-del-mundo"]

In [45]:
#ii. Lectores.
resumen_nulos_lectores = pd.DataFrame({
    "nulos": df_lectores.isnull().sum(),
    "pct_nulos": (df_lectores.isnull().sum() / len(df_lectores) * 100).round(2)
})
resumen_nulos_lectores

,nulos,pct_nulos
id_lector,0,0.0
nombre,0,0.0
genero,0,0.0
vive_en,0,0.0
nacimiento,0,0.0


In [46]:
#iii. Interacciones.
resumen_nulos_interacciones = pd.DataFrame({
    "nulos": df_interacciones.isnull().sum(),
    "pct_nulos": (df_interacciones.isnull().sum() / len(df_interacciones) * 100).round(2)
})
resumen_nulos_interacciones

,nulos,pct_nulos
id_lector,0,0.0
id_libro,0,0.0
fecha,0,0.0
rating,0,0.0


In [47]:
#e. Tipos de datos.
#i. Libros.
#1. Imprimo.
print(df_libros.dtypes)

#2. Convierto anio_edicion
df_libros = limpiar_anio_edicion(df_libros,anio_actual)

id_libro        object
titulo          object
autor           object
genero          object
editorial       object
anio_edicion    object
isbn            object
resumen         object
img_src         object
dtype: object
Nulos tras extraer año válido: 78305
Años fuera de rango [1800, 2024] nulificados: 717
Nulos tras imputar por promedio de título: 78969
Nulos tras imputar por promedio de editorial: 78275
Nulos tras imputar por mediana general: 0


In [48]:
#ii. Lectores.
#1. Imprimo.
print(df_lectores.dtypes)

#2. Convierto Nacimiento.
df_lectores = limpiar_nacimiento(df_lectores,anio_actual)

id_lector     object
nombre        object
genero        object
vive_en       object
nacimiento    object
dtype: object
Nulos tras forzar a numérico: 3429
Nacimientos fuera de rango [1900, 2024] nulificados: 0
Nulos tras imputar por promedio de nombre: 3167
Nulos tras imputar por promedio global: 0
Nulos restantes: 0


In [49]:
#iii. Interacciones.
#1. Imprimo.
print(df_interacciones.dtypes)

#2. Corrijo Fecha.
#a. Convierto a datetime, lo que no matchee queda NaT.
fecha_original_nulos = df_interacciones["fecha"].isnull().sum()
df_interacciones["fecha"] = pd.to_datetime(df_interacciones["fecha"], format="%d-%m-%Y", errors="coerce")
print(f"Fechas no parseables (NaT) tras convertir: {df_interacciones['fecha'].isnull().sum() - fecha_original_nulos}")

#b. Saco las filas con fecha no parseable (no tiene sentido imputar una fecha de interacción).
filas_antes = len(df_interacciones)
df_interacciones = df_interacciones.dropna(subset=["fecha"])
print(f"Filas eliminadas por fecha inválida: {filas_antes - len(df_interacciones)}")

id_lector    object
id_libro     object
fecha        object
rating        int64
dtype: object
Fechas no parseables (NaT) tras convertir: 1
Filas eliminadas por fecha inválida: 1


In [50]:
#f. Dataset a predecir.
#i. Cantidad de lectores.
cant_lectores_predecir = len(df_entregar["id_lector"].unique())
print("Hay {} lectores únicos a predecir".format(cant_lectores_predecir))

#ii. Cantidad de lectores a predecir que no están en df_lectores (lectores "nuevos", sin atributos conocidos).
lectores_predecir = df_entregar["id_lector"].unique()
lectores_conocidos = df_lectores["id_lector"].unique()

lectores_sin_datos = set(lectores_predecir) - set(lectores_conocidos)
print(f"Hay {len(lectores_sin_datos)} lectores a predecir que NO están en df_lectores.")

#iii. Cantidad de lectores a predecir que si están en df_lectores (para tener el panorama completo).
lectores_con_datos = set(lectores_predecir) & set(lectores_conocidos)
print(f"Hay {len(lectores_con_datos)} lectores a predecir que SÍ están en df_lectores.")

#iv. De los lectores que si están en df_lectores, reviso nulos por columna (¿tienen datos completos o les faltan atributos?).
df_lectores_con_datos = df_lectores[df_lectores["id_lector"].isin(lectores_con_datos)]

resumen_nulos_lectores_predecir = pd.DataFrame({
    "nulos": df_lectores_con_datos.isnull().sum(),
    "pct_nulos": (df_lectores_con_datos.isnull().sum() / len(df_lectores_con_datos) * 100).round(2)
})
print(f"Revisando nulos sobre {len(df_lectores_con_datos)} lectores (los que están en df_lectores y hay que predecirles).")
display(resumen_nulos_lectores_predecir)

#v. Cantidad de lectores a predecir con interacciones (o sea, que tienen historial en df_interacciones).
mask_interacciones_predecir = df_interacciones["id_lector"].isin(lectores_predecir)
lectores_predecir_con_interacciones = df_interacciones.loc[mask_interacciones_predecir, "id_lector"].unique()

print(f"Lectores a predecir con al menos 1 interacción: {len(lectores_predecir_con_interacciones)} de {len(lectores_predecir)}")

#vi. Cantidad de interacciones promedio por lector a predecir.
interacciones_por_lector_predecir = df_interacciones[mask_interacciones_predecir].groupby("id_lector").size()

print(f"Promedio de interacciones por lector a predecir: {interacciones_por_lector_predecir.mean():.2f}")
print(f"Mediana: {interacciones_por_lector_predecir.median():.0f}")
print(f"Mínimo: {interacciones_por_lector_predecir.min()}")
print(f"Máximo: {interacciones_por_lector_predecir.max()}")


Hay 832 lectores únicos a predecir
Hay 1 lectores a predecir que NO están en df_lectores.
Hay 831 lectores a predecir que SÍ están en df_lectores.
Revisando nulos sobre 831 lectores (los que están en df_lectores y hay que predecirles).


,nulos,pct_nulos
id_lector,0,0.0
nombre,0,0.0
genero,0,0.0
vive_en,0,0.0
nacimiento,0,0.0


Lectores a predecir con al menos 1 interacción: 816 de 832
Promedio de interacciones por lector a predecir: 162.22
Mediana: 95
Mínimo: 1
Máximo: 2260


#### 6. Análisis Exploratorio de Datos (EDA) y Correcciones.

In [51]:
#a. Libros.
#1. Autor: distribución y pareto (tabla).
dist_autor = df_libros["autor"].value_counts().reset_index()
dist_autor.columns = ["autor", "cantidad"]
dist_autor["pct"] = (dist_autor["cantidad"] / dist_autor["cantidad"].sum() * 100).round(2)
dist_autor["pct_acumulado"] = dist_autor["pct"].cumsum().round(2)

cant_autores_total = dist_autor["autor"].nunique()
cant_autores_80pct = (dist_autor["pct_acumulado"] <= 80).sum() + 1  # +1 para incluir el autor que cruza el 80%

print(f"Cantidad total de autores: {cant_autores_total}")
print(f"Cantidad de autores que explican el 80% de los libros: {cant_autores_80pct} ({(cant_autores_80pct / cant_autores_total * 100):.2f}% del total de autores)")
print("Pareto de autores (top 15):")
display(dist_autor.head(15))

Cantidad total de autores: 21655
Cantidad de autores que explican el 80% de los libros: 21656 (100.00% del total de autores)
Pareto de autores (top 15):


,autor,cantidad,pct,pct_acumulado
0,VV.AA.,475,0.94,0.94
1,"SIERRA I FABRA, JORDI",121,0.24,1.18
2,"KING, STEPHEN",111,0.22,1.40
3,"ROBERTS, NORA",109,0.22,1.62
4,"ASIMOV, ISAAC",103,0.20,1.82
5,"CHRISTIE, AGATHA",96,0.19,2.01
6,"PÉREZ GALDÓS, BENITO",93,0.18,2.19
7,"VÁZQUEZ-FIGUEROA, ALBERTO",78,0.15,2.34
8,"VIDAL, CÉSAR",76,0.15,2.49
9,"CAMILLERI, ANDREA",76,0.15,2.64


In [52]:
#2. Género: distribución, pareto (tabla) y gráfico.
dist_genero = df_libros["genero"].value_counts().reset_index()
dist_genero.columns = ["genero", "cantidad"]
dist_genero["pct"] = (dist_genero["cantidad"] / dist_genero["cantidad"].sum() * 100).round(2)
dist_genero["pct_acumulado"] = dist_genero["pct"].cumsum().round(2)

cant_generos_total = dist_genero["genero"].nunique()
cant_generos_80pct = (dist_genero["pct_acumulado"] <= 80).sum() + 1

print(f"Cantidad total de géneros: {cant_generos_total}")
print(f"Cantidad de géneros que explican el 80% de los libros: {cant_generos_80pct} ({(cant_generos_80pct / cant_generos_total * 100):.2f}% del total de géneros)")
print("Pareto de géneros (top 15):")
display(dist_genero.head(15))

Cantidad total de géneros: 66
Cantidad de géneros que explican el 80% de los libros: 10 (15.15% del total de géneros)
Pareto de géneros (top 15):


,genero,cantidad,pct,pct_acumulado
0,Narrativa,8638,17.10,17.10
1,"Novela negra, intriga, terror",6874,13.61,30.71
2,Ensayo,4472,8.85,39.56
3,Infantil y juvenil,3973,7.86,47.42
4,Histórica y aventuras,3876,7.67,55.09
5,"Fantástica, ciencia ficción",2999,5.94,61.03
6,"Romántica, erótica",2953,5.84,66.87
7,Ficción literaria,2750,5.44,72.31
8,Literatura contemporánea,2601,5.15,77.46
9,"Cómics, Novela Gráfica",2197,4.35,81.81


In [53]:
#3. Editorial: distribución y pareto (tabla).
dist_editorial = df_libros["editorial"].value_counts().reset_index()
dist_editorial.columns = ["editorial", "cantidad"]
dist_editorial["pct"] = (dist_editorial["cantidad"] / dist_editorial["cantidad"].sum() * 100).round(2)
dist_editorial["pct_acumulado"] = dist_editorial["pct"].cumsum().round(2)

cant_editoriales_total = dist_editorial["editorial"].nunique()
cant_editoriales_80pct = (dist_editorial["pct_acumulado"] <= 80).sum() + 1

print(f"Cantidad total de editoriales: {cant_editoriales_total}")
print(f"Cantidad de editoriales que explican el 80% de los libros: {cant_editoriales_80pct} ({(cant_editoriales_80pct / cant_editoriales_total * 100):.2f}% del total de editoriales)")
print("Pareto de editoriales (top 15):")
display(dist_editorial.head(15))

Cantidad total de editoriales: 2909
Cantidad de editoriales que explican el 80% de los libros: 159 (5.47% del total de editoriales)
Pareto de editoriales (top 15):


,editorial,cantidad,pct,pct_acumulado
0,PLANETA,1965,3.89,3.89
1,EDICIONES B,1830,3.62,7.51
2,DEBOLSILLO,1750,3.46,10.97
3,ALFAGUARA,1562,3.09,14.06
4,ANAGRAMA,1489,2.95,17.01
5,ALIANZA,1324,2.62,19.63
6,AUTOR-EDITOR,1210,2.40,22.03
7,PLAZA & JANÉS,1085,2.15,24.18
8,DESTINO,1006,1.99,26.17
9,TUSQUETS,920,1.82,27.99


In [54]:
#4. Año de edición: distribución, pareto (tabla) y gráfico.
anio_edicion_num = pd.to_numeric(df_libros["anio_edicion"], errors="coerce")

dist_anio = anio_edicion_num.value_counts().reset_index()
dist_anio.columns = ["anio_edicion", "cantidad"]
dist_anio = dist_anio.sort_values("cantidad", ascending=False).reset_index(drop=True)
dist_anio["pct"] = (dist_anio["cantidad"] / dist_anio["cantidad"].sum() * 100).round(2)
dist_anio["pct_acumulado"] = dist_anio["pct"].cumsum().round(2)

cant_anios_total = dist_anio["anio_edicion"].nunique()
cant_anios_80pct = (dist_anio["pct_acumulado"] <= 80).sum() + 1

print(f"Cantidad total de años de edición distintos: {cant_anios_total}")
print(f"Cantidad de años que explican el 80% de los libros: {cant_anios_80pct} ({(cant_anios_80pct / cant_anios_total * 100):.2f}% del total de años)")
print("Pareto de años de edición (top 15):")
display(dist_anio.head(15))

#5. Año de edición: detección de valores raros/atípicos.
#i. Rango general y estadísticas descriptivas.
print("Estadísticas de anio_edicion (numérico):")
display(anio_edicion_num.describe())

#ii. Resumen de rangos (bins por siglo/período) para dimensionar cada bloque.
bins = [-float("inf"), 0, 1800, 1900, 1950, 2000, anio_actual, float("inf")]
labels = ["<=0", "1-1799", "1800-1899", "1900-1949", "1950-1999", "2000-actual", "post-actual"]
dist_rangos = pd.cut(anio_edicion_num, bins=bins, labels=labels).value_counts().reindex(labels)
print("\nCantidad de libros por rango de año:")
display(dist_rangos)

#iii. Libros con año == 0 o negativo (posibles placeholders de "sin dato").
libros_placeholder = df_libros[anio_edicion_num <= 0]
print(f"\nLibros con año <= 0 (posible placeholder): {len(libros_placeholder)}")
display(libros_placeholder[["id_libro", "titulo", "autor", "anio_edicion"]].head(15))

Cantidad total de años de edición distintos: 111
Cantidad de años que explican el 80% de los libros: 10 (9.01% del total de años)
Pareto de años de edición (top 15):


,anio_edicion,cantidad,pct,pct_acumulado
0,2011,81123,63.01,63.01
1,2009,2953,2.29,65.30
2,2008,2863,2.22,67.52
3,2010,2720,2.11,69.63
4,2007,2655,2.06,71.69
5,2012,2564,1.99,73.68
6,2013,2517,1.96,75.64
7,2015,2220,1.72,77.36
8,2017,2212,1.72,79.08
9,2014,2146,1.67,80.75


Estadísticas de anio_edicion (numérico):


count    128743.000000
mean       2010.572016
std           5.975342
min        1835.000000
25%        2011.000000
50%        2011.000000
75%        2011.000000
max        2024.000000
Name: anio_edicion, dtype: float64


Cantidad de libros por rango de año:


anio_edicion
<=0                 0
1-1799              0
1800-1899          12
1900-1949          53
1950-1999        5281
2000-actual    123397
post-actual         0
Name: count, dtype: int64


Libros con año <= 0 (posible placeholder): 0


,id_libro,titulo,autor,anio_edicion


In [55]:
#6. Resumen: longitud promedio, mínima y máxima (tabla resumen).
longitud_resumen = df_libros["resumen"].dropna().str.len()

resumen_longitud = pd.DataFrame({
    "métrica": ["longitud_promedio", "longitud_minima", "longitud_maxima"],
    "valor": [longitud_resumen.mean().round(2), longitud_resumen.min(), longitud_resumen.max()]
})
print("Longitud de resúmenes (en caracteres):")
display(resumen_longitud)

Longitud de resúmenes (en caracteres):


,métrica,valor
0,longitud_promedio,723.49
1,longitud_minima,0.00
2,longitud_maxima,4523.00


In [56]:
#b. Lectores.
#1. Género: distribución (tabla y gráfico).
dist_genero_lector = df_lectores["genero"].value_counts(dropna=False).reset_index()
dist_genero_lector.columns = ["genero", "cantidad"]
dist_genero_lector["pct"] = (dist_genero_lector["cantidad"] / dist_genero_lector["cantidad"].sum() * 100).round(2)
print("Distribución de lectores por género:")
display(dist_genero_lector)

Distribución de lectores por género:


,genero,cantidad,pct
0,Mujer,3925,34.78
1,Hombre,3712,32.89
2,-,3648,32.33


In [57]:
#2. Vive_en: distribución y pareto (tabla), sobre el dato crudo.
dist_vive_en = df_lectores["vive_en"].value_counts(dropna=False).reset_index()
dist_vive_en.columns = ["vive_en", "cantidad"]
dist_vive_en["pct"] = (dist_vive_en["cantidad"] / dist_vive_en["cantidad"].sum() * 100).round(2)
dist_vive_en["pct_acumulado"] = dist_vive_en["pct"].cumsum().round(2)

cant_ubicaciones_total = dist_vive_en["vive_en"].nunique()
cant_ubicaciones_80pct = (dist_vive_en["pct_acumulado"] <= 80).sum() + 1

print(f"Cantidad total de ubicaciones distintas: {cant_ubicaciones_total}")
print(f"Cantidad de ubicaciones que explican el 80% de los lectores: {cant_ubicaciones_80pct} ({(cant_ubicaciones_80pct / cant_ubicaciones_total * 100):.2f}% del total de ubicaciones)")
print("Pareto de ubicaciones (top 15):")
display(dist_vive_en.head(15))

Cantidad total de ubicaciones distintas: 1585
Cantidad de ubicaciones que explican el 80% de los lectores: 129 (8.14% del total de ubicaciones)
Pareto de ubicaciones (top 15):


,vive_en,cantidad,pct,pct_acumulado
0,España,3068,27.19,27.19
1,Madrid - España,969,8.59,35.78
2,,534,4.73,40.51
3,¿?,457,4.05,44.56
4,Barcelona - España,377,3.34,47.90
5,Valencia - España,213,1.89,49.79
6,Sevilla - España,199,1.76,51.55
7,Buenos aires - Argentina,159,1.41,52.96
8,Mexico,155,1.37,54.33
9,Argentina,135,1.20,55.53


In [58]:
#3. Vive_en: separo en pais y ciudad, limpiando placeholders.
#a. Separo por " - ": si hay 2 partes, la primera es ciudad y la segunda país.
vive_en_str = df_lectores["vive_en"].astype(str).str.strip()
vive_en_str = vive_en_str.replace("nan", np.nan)

#i. Caso especial: strings que terminan en "-" (el país quedó vacío tras el strip, ej. "Madrid -").
termina_en_guion = vive_en_str.str.endswith("-", na=False)
vive_en_str = vive_en_str.where(~termina_en_guion, vive_en_str + " ")

partes = vive_en_str.str.split(" - ", n=1, expand=True)

ciudad_raw = np.where(partes[1].notna(), partes[0].str.strip(), np.nan)
pais_raw = np.where(partes[1].notna(), partes[1].str.strip(), partes[0].str.strip())

df_lectores["ciudad"] = pd.Series(ciudad_raw, index=df_lectores.index)
df_lectores["pais"] = pd.Series(pais_raw, index=df_lectores.index)

#b. Limpio placeholders (vacíos, variantes de "¿?") en CADA columna resultante.
patron_placeholder = r"^\s*$|^\W*\?+\W*$"
df_lectores["ciudad"] = df_lectores["ciudad"].replace(patron_placeholder, np.nan, regex=True)
df_lectores["pais"] = df_lectores["pais"].replace(patron_placeholder, np.nan, regex=True)

#c. Normalizo texto: minúsculas y sin tildes, para consolidar duplicados (ej. "Peru" vs "Perú", "Argentina" vs "ARGENTINA").
#   Muevo este paso ACÁ (antes vivía en la celda de corrección) para que la inferencia y el agrupamiento de más abajo trabajen sobre texto ya consolidado.
for col in ["ciudad", "pais"]:
    df_lectores[col] = df_lectores[col].str.lower().apply(quitar_tildes)

#d. Reviso resultado.
print(f"Nulos en pais: {df_lectores['pais'].isnull().sum()}")
print(f"Nulos en ciudad: {df_lectores['ciudad'].isnull().sum()}")
print("\nDistribución de país (top 15):")
display(df_lectores["pais"].value_counts(dropna=False).head(15))
print("\nDistribución de ciudad (top 15):")
display(df_lectores["ciudad"].value_counts(dropna=False).head(15))

Nulos en pais: 1082
Nulos en ciudad: 4546

Distribución de país (top 15):


pais
espana                      7671
NaN                         1082
mexico                       657
argentina                    636
colombia                     262
cote d'ivoire                168
venezuela                    138
peru                         125
chile                        119
uruguay                       58
united states of america      49
ecuador                       39
guatemala                     25
panama                        21
cuba                          17
Name: count, dtype: int64


Distribución de ciudad (top 15):


ciudad
NaN             4546
madrid           982
barcelona        387
valencia         223
sevilla          200
buenos aires     162
santiago         152
malaga           124
bogota            97
granada           90
cordoba           83
lima              82
zaragoza          81
murcia            79
alicante          65
Name: count, dtype: int64

In [59]:
#4. Nacimiento: estadísticas descriptivas y detección de valores raros.
nacimiento_num = pd.to_numeric(df_lectores["nacimiento"], errors="coerce")

print("Estadísticas de nacimiento (numérico):")
display(nacimiento_num.describe())

#a. Valores no numéricos.
mask_no_numerico = df_lectores["nacimiento"].notna() & nacimiento_num.isna()
print(f"\nValores no numéricos en nacimiento: {mask_no_numerico.sum()}")

#b. Lectores con nacimiento implausible (muy viejo, ej. antes de 1900) o futuro.
lectores_raros = df_lectores[(nacimiento_num < 1900) | (nacimiento_num > anio_actual)]
print(f"\nLectores con nacimiento < 1900 o > {anio_actual}: {len(lectores_raros)}")
display(lectores_raros[["id_lector", "nombre", "genero", "vive_en", "nacimiento"]].head(15))

#c. Lectores "menores" implausibles (nacimiento muy reciente, edad < 5 años).
lectores_muy_jovenes = df_lectores[nacimiento_num > anio_actual - 5]
print(f"\nLectores con nacimiento posterior a {anio_actual - 5} (edad < 5 años): {len(lectores_muy_jovenes)}")
display(lectores_muy_jovenes[["id_lector", "nombre", "genero", "vive_en", "nacimiento"]].head(15))

Estadísticas de nacimiento (numérico):


count    11285.000000
mean      1978.099424
std         17.092079
min       1910.000000
25%       1977.000000
50%       1978.000000
75%       1988.000000
max       2013.000000
Name: nacimiento, dtype: float64


Valores no numéricos en nacimiento: 0

Lectores con nacimiento < 1900 o > 2024: 0


,id_lector,nombre,genero,vive_en,nacimiento



Lectores con nacimiento posterior a 2019 (edad < 5 años): 0


,id_lector,nombre,genero,vive_en,nacimiento


In [60]:
#c. Interacciones.
#1. Shape y tipos.
print(f"Shape: {df_interacciones.shape}")
print(df_interacciones.dtypes)

Shape: (461407, 4)
id_lector            object
id_libro             object
fecha        datetime64[ns]
rating                int64
dtype: object


In [61]:
#2. Rating: estadísticas descriptivas y distribución.
print("\nEstadísticas de rating:")
display(df_interacciones["rating"].describe())

dist_rating = df_interacciones["rating"].value_counts(dropna=False).sort_index().reset_index()
dist_rating.columns = ["rating", "cantidad"]
dist_rating["pct"] = (dist_rating["cantidad"] / dist_rating["cantidad"].sum() * 100).round(2)
print("\nDistribución de rating:")
display(dist_rating)


Estadísticas de rating:


count    461407.000000
mean          7.263399
std           1.824603
min           1.000000
25%           6.000000
50%           7.000000
75%           8.000000
max          10.000000
Name: rating, dtype: float64


Distribución de rating:


,rating,cantidad,pct
0,1,5173,1.12
1,2,5124,1.11
2,3,4567,0.99
3,4,18870,4.09
4,5,25328,5.49
5,6,87527,18.97
6,7,85018,18.43
7,8,122290,26.50
8,9,53567,11.61
9,10,53943,11.69


In [62]:
#3. Rating: nulos y valores fuera de rango esperado (asumiendo escala 1-10 según la muestra).
print(f"\nNulos en rating: {df_interacciones['rating'].isnull().sum()}")
rating_fuera_rango = df_interacciones[(df_interacciones["rating"] < 1) | (df_interacciones["rating"] > 10)]
print(f"Ratings fuera de rango [1, 10]: {len(rating_fuera_rango)}")
display(rating_fuera_rango.head(15))


Nulos en rating: 0
Ratings fuera de rango [1, 10]: 0


,id_lector,id_libro,fecha,rating


In [63]:
#4. Fecha: conversión a datetime y detección de valores no parseables.
fecha_dt = pd.to_datetime(df_interacciones["fecha"], errors="coerce")
mask_fecha_invalida = df_interacciones["fecha"].notna() & fecha_dt.isna()
print(f"\nFechas no parseables: {mask_fecha_invalida.sum()}")
if mask_fecha_invalida.sum() > 0:
    display(df_interacciones.loc[mask_fecha_invalida, "fecha"].value_counts().head(15))


Fechas no parseables: 0


In [64]:
#5. Fecha: rango temporal y valores extremos.
print(f"\nFecha mínima: {fecha_dt.min()}")
print(f"Fecha máxima: {fecha_dt.max()}")
print(f"Nulos en fecha (original): {df_interacciones['fecha'].isnull().sum()}")

limite_fecha = pd.Timestamp(year=anio_actual, month=12, day=31)
fechas_futuras = df_interacciones[fecha_dt > limite_fecha]
print(f"\nInteracciones con fecha futura: {len(fechas_futuras)}")
display(fechas_futuras.head(15))


Fecha mínima: 2008-02-24 00:00:00
Fecha máxima: 2024-12-31 00:00:00
Nulos en fecha (original): 0

Interacciones con fecha futura: 0


,id_lector,id_libro,fecha,rating


#### 7. EDA sobre el dataframe mergeado.

In [ ]:
#a. Mergeo las tablas.
#i. Renombro columnas para que no sea confuso luego del merge su origen.
df_libros.rename({
    "genero":"genero_libro",
    "genero_agrupado":"genero_libro_agrupado"
},axis=1,inplace=True)

df_lectores.rename({
    "genero":"genero_persona",
    "pais":"pais_persona",
    "pais_agrupado":"pais_persona_agrupado",
    "ciudad":"ciudad_persona"
},axis=1,inplace=True)

#ii. Mergeo.
df_total = df_interacciones.merge(df_libros,how="left",on="id_libro").merge(df_lectores,how="left",on="id_lector")

In [ ]:
#b. Selecciono columnas de interés.
df_total = df_total[['id_lector', 'id_libro', 'fecha', 'rating',                              # Interacciones.
                    'autor', 'genero_libro_agrupado', 'editorial_agrupada', 'anio_edicion',   # Libros.
                    'genero_persona', 'pais_persona_agrupado', 'nacimiento']]                 # Personas.
# Nota: saco "ciudad_persona" (demasiada cardinalidad para dummies), agrego "autor" (lo necesito para features lector x autor).

In [ ]:
#c. Me fijo combinaciones únicas para, también, entender si me conviene sacar promedios, o no.

#i. Lector - Libro.
combos_libro = df_total[["id_lector","id_libro"]].drop_duplicates()
interacciones_por_lector_libro = combos_libro.groupby("id_lector").size()

#ii. Lector - Autor.
combos_autor = df_total[["id_lector","autor"]].drop_duplicates()
autores_por_lector = combos_autor.groupby("id_lector").size()

repeticion_autor = (
    df_total.groupby(["id_lector","autor"]).size().rename("n_interacciones").reset_index()
)
repeticion_autor["repetido"] = repeticion_autor["n_interacciones"] > 1
pct_autores_repetidos_por_lector = repeticion_autor.groupby("id_lector")["repetido"].mean()

#iii. Lector - Género de libro.
combos_genero = df_total[["id_lector","genero_libro_agrupado"]].drop_duplicates()
generos_por_lector = combos_genero.groupby("id_lector").size()

repeticion_genero = (
    df_total.groupby(["id_lector","genero_libro_agrupado"]).size().rename("n_interacciones").reset_index()
)
repeticion_genero["repetido"] = repeticion_genero["n_interacciones"] > 1
pct_generos_repetidos_por_lector = repeticion_genero.groupby("id_lector")["repetido"].mean()

#d. Resumen consolidado: cuántos items distintos, y qué % de ellos repite.
resumen = pd.DataFrame({
    "libro_distintos_leidos_por_lector":  interacciones_por_lector_libro.describe(),
    "autor_distintos_leidos_por_lector":  autores_por_lector.describe(),
    "genero_distintos_leidos_por_lector": generos_por_lector.describe(),
    #"libro_pct_repetido":     pct_libros_repetidos_por_lector.describe(),
    "pct_lector_repite_autor":     pct_autores_repetidos_por_lector.describe(),
    "pct_lector_repite_genero":    pct_generos_repetidos_por_lector.describe(),
}).T

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
resumen[["mean","50%","std","min","max"]].rename(columns={"50%":"mediana"})

,mean,mediana,std,min,max
libro_distintos_leidos_por_lector,43.231,9.000,101.129,1.000,"2,260.000"
autor_distintos_leidos_por_lector,26.037,7.000,55.350,1.000,951.000
genero_distintos_leidos_por_lector,5.706,4.000,4.652,1.000,24.000
pct_lector_repite_autor,0.198,0.167,0.227,0.000,1.000
pct_lector_repite_genero,0.492,0.500,0.354,0.000,1.000
